## Import Text

The text being imported is called 'the verdict', it's just a txt file with no specialty in it.

In [3]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/" "LLMs-from-scratch/main/ch02/01_main-chapter-code/" "the-verdict.txt")

    file_path = "the-verdict.txt"
    response = requests.get(url)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


we can look at what is inside the text

In [5]:
with open(file_path, "r") as f:
    text = f.read()

print(len(text))
print(text[:50] + '...')

20479
I HAD always thought Jack Gisburn rather a cheap g...


## Regular Expressions Tokenization


And an extremely simply way of tokenization, is through splitting the sentence, or through regular expressions.

In [13]:
# splitting a sentence

short_text = text[:50]
print(short_text)
stripped_short_text = short_text.split(" ")
print(stripped_short_text)


# regualr expressions

import re

print()
re_short_text = re.split(r'([,.]|\s)', short_text)
re_short_text = [txt.strip() for txt in re_short_text] # Getting rid of white space
print(re_short_text)

I HAD always thought Jack Gisburn rather a cheap g
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'g']

['I', '', 'HAD', '', 'always', '', 'thought', '', 'Jack', '', 'Gisburn', '', 'rather', '', 'a', '', 'cheap', '', 'g']


The author went deeper with regular expressions to make a basic tokenizor, and... I'm not really a fan.

In [15]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
4690


But now, we'll use that implementation and focus on building a vocabulary, and a simple tokenizer

In [27]:
# set removes reundencies, sorted makes it alphabetical ordered
unique_words = sorted(set(preprocessed))

vocab_size = len(unique_words)
print(vocab_size)

1130


In [28]:
vocab = {token: index_integer for index_integer, token in enumerate(unique_words)}

for item in vocab.items():
    print(item)
    break

('!', 0)


## Making Tokenizer

from such, we can make a class for a simple tokenizer.

In [39]:
class TokenizerV1:

    def __init__(self, vocab):
        self.vocab = vocab
        self.inverse_vocab = {index_integer: token for token, index_integer in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        encoded_ids = [self.vocab[text] for text in preprocessed]
        return encoded_ids
        
    def decode(self, ids):
        decoded_text = " ".join([self.inverse_vocab[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', decoded_text)
        return text


In [40]:
v1_tok = TokenizerV1(vocab)

random_text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

ids = v1_tok.encode(random_text)
print(ids)
decoded_text = v1_tok.decode(ids)
print(decoded_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


We could add additional tokens in the vocabulary for the purpose of an LLM

In [29]:
vocab.update({"<|endoftext|>": len(vocab), "<|unk|>": len(vocab) + 1})

for item in list(vocab.items())[-5:]:
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


This will create our 2nd simple tokenizer.

In [50]:
class TokenizerV2:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inverse_vocab = {index_integer: token for token, index_integer in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.vocab else "<|unk|>" for item in preprocessed] # adds a check for unknown strings
        preprocessed.append("<|endoftext|>") # adds an end of text token
        ids = [self.vocab[s] for s in preprocessed] 
        return ids
        
    def decode(self, ids):
        text = " ".join([self.inverse_vocab[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [51]:
tokenizer = TokenizerV2(vocab)

text = "Hello, do you like tea? In the sunlit terraces of the palace."

encoded_text = tokenizer.encode(text)
print(encoded_text)

decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1131, 5, 355, 1126, 628, 975, 10, 55, 988, 956, 984, 722, 988, 1131, 7, 1130]
<|unk|>, do you like tea? In the sunlit terraces of the <|unk|>. <|endoftext|>


### BPE


You can go deeper with regular expressions, but they won't be able to deal with te nuance of all language, such as:

- emojis 🤔
- other languages 你好

As Andrew Kaparthy said it himself, Tokenization is the heart of so many problems in LLMs...

<img src="attachments/1.png" width="500px">

and so we'll move on to using a library/algorithm called BPE, the actual tokenization method used by GPT.

In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base") # the original repo used gpt 2

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)

print(strings)

[9906, 11, 656, 499, 1093, 15600, 30, 220, 100257, 763, 279, 7160, 32735, 7317, 2492]
Hello, do you like tea? <|endoftext|> In the sunlit terraces


Now we will start preparing the training text data for the pre-training of the LLM with that verdict file.

In [6]:
with open("the-verdict.txt", "r") as f:
    text = f.read()
enc_text = tokenizer.encode(text)

this is a small sliding window example

In [5]:
context_size = 8

data = enc_text[:context_size]

for index in range(1, context_size):
    an_input = data[:index]
    an_output = data[index]
    print(an_input, "->", an_output)

[40] -> 473
[40, 473] -> 1846
[40, 473, 1846] -> 2744
[40, 473, 1846, 2744] -> 3463
[40, 473, 1846, 2744, 3463] -> 7762
[40, 473, 1846, 2744, 3463, 7762] -> 480
[40, 473, 1846, 2744, 3463, 7762, 480] -> 285


We will move on to implement the entire data set in pytorch.

In [36]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDatasetV1(Dataset):

    # oh reminds me of the good old days of making Pytorch stuff...
    def __init__(self, text, tokenizer, context_length, stride):
        super().__init__()
        self.input_ids = []
        self.output_ids = []

        # tokenize the text
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # divide the text down into context_length size 
        for index in range(0, len(token_ids) - context_length, stride):
            inputs = token_ids[index:index + context_length]
            outputs = token_ids[index+1:index + context_length + 1]
            self.input_ids.append(inputs)
            self.output_ids.append(outputs)

    
    # functions needed by a map type pytorch dataloader
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return torch.tensor(self.input_ids[index]), torch.tensor(self.output_ids[index])


In [37]:
# Hyperparameters
batch_size = 4
context_length = 256
stride = 128
shuffle = True
drop_last = True # The last batch likely is not full size
num_workers = 0 # GPU related stuff, we are on a mac

tokenizer = tiktoken.get_encoding("cl100k_base")

dataset = TextDatasetV1(text, tokenizer, context_length, stride)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,drop_last=drop_last, num_workers=num_workers)

In [39]:
first_batch = next(iter(dataloader))
print(first_batch[0])
print(first_batch[1])

tensor([[  364,  3195,   709,  ...,   311,   480,   285],
        [ 6548,  1436,  1518,  ...,  5046,  3388,   358],
        [ 1139,   264,  6453,  ...,   382,     1,  1383],
        [ 2181,   596,   813,  ...,  2539,   315, 18718]])
tensor([[ 3195,   709,    26,  ...,   480,   285, 22464],
        [ 1436,  1518,  1555,  ...,  3388,   358,  3287],
        [  264,  6453, 14733,  ...,     1,  1383,   622],
        [  596,   813, 27873,  ...,   315, 18718, 12657]])


## Token Embeddings

Starting with word embeddings

In [33]:
encoding = tiktoken.get_encoding("cl100k_base")
vocab_size = len(encoding._mergeable_ranks)
embedding_size = 728

word_embeddings = torch.nn.Embedding(vocab_size, embedding_size)
print(word_embeddings)

Embedding(100256, 728)


and then positional embeddings, which relies on context length.

In [34]:
pos_embeddings = torch.nn.Embedding(context_length, embedding_size)
print(pos_embeddings)

Embedding(256, 728)


and this would be a single pass of data: word embedding + pos embeddings

In [40]:
input_embeddings = word_embeddings(first_batch[0]) + pos_embeddings(torch.arange(context_length))
print(input_embeddings.shape)

torch.Size([4, 256, 728])
